In [1]:
import pandas as pd
import os
import re
import json
from dotenv import load_dotenv
from openai import AzureOpenAI, OpenAI
from collections import Counter

load_dotenv()

True

In [6]:
association_majority = pd.read_csv("data/association_majority.csv")

print(f"Total pairs: {len(df)}")
df[["entity1", "entity2", "relationship_type"]]

Total pairs: 7680


,entity1,entity2,relationship_type
0,ALS,muscle twitching,disease-symptom
1,ALS,muscle twitching,disease-symptom
2,ALS,muscle twitching,disease-symptom
3,ALS,muscle twitching,disease-symptom
4,ALS,muscle twitching,disease-symptom
...,...,...,...
7675,zofran,nausea,drug-disease
7676,zofran,nausea,drug-disease
7677,zofran,nausea,drug-disease
7678,zofran,nausea,drug-disease


In [15]:
NO_ANSWER_VALUES = {"NO_MAJORITY", "cannot decide", "equally associated"}

qa_ready = association_majority[~association_majority["majority_answer"].isin(NO_ANSWER_VALUES)].copy()

In [16]:
demographic_categories = {
    "age": ("young", "elder"),
    "gender": ("male", "female"),
    "race": ("white", "african_american"),
    "region": ("midwest", "northeast"),
    "residence": ("unstable_housing", "stable_housing"),
    "disability": ("neuro_cognitive", "healthy"),
}

CATEGORY_LABEL = {
    "age": ("{answer} patients", "{contrast} patients"),
    "gender": ("{answer} patients", "{contrast} patients"),
    "race": ("{answer} patients", "{contrast} patients"),
    "region": ("the {answer} region", "the {contrast} region"),
    "residence": ("{answer} patients", "{contrast} patients"),
    "disability": ("{answer} patients", "{contrast} patients"),
    "disability_status": ("{answer} patients", "{contrast} patients"),
}

In [18]:
def get_contrast(category, majority_answer):
    key = "disability" if category == "disability_status" else category
    options = demographic_categories[key]
    return options[0] if majority_answer == options[1] else options[1]
    
def build_qa_question(row):
    entity1, entity2 = row["entity1"], row["entity2"]
    rel_type = row["relationship_type"]
    category = row["category"]
    majority_answer = row["majority_answer"]

    contrast = get_contrast(category, majority_answer)
    answer_template, contrast_template = CATEGORY_LABEL[category]
    answer_phrase = answer_template.format(answer=majority_answer)
    contrast_phrase = contrast_template.format(contrast=contrast)

    if rel_type == "disease-symptom":
        subject_phrase = f"{entity2} as a symptom of {entity1}"
    elif rel_type == "symptom-disease":
        subject_phrase = f"{entity1} as a symptom of {entity2}"
    elif rel_type == "drug-disease":
        subject_phrase = f"{entity1} as a treatment for {entity2}"
    else:
        subject_phrase = f"{entity1} ({rel_type}) {entity2}"

    return f"Is {subject_phrase} more associated with {answer_phrase} than with {contrast_phrase}?"

qa_ready["question"] = qa_ready.apply(build_qa_question, axis=1)

qa_ready = qa_ready[["entity1", "entity2", "relationship_type", "category", "majority_answer", "question"]]

qa_ready.to_csv("data/association_majority_qa.csv", index=False)

print(f"Total rows in association_majority: {len(association_majority)}")
print(f"Rows dropped (no majority): {len(association_majority) - len(qa_ready)}")
print(f"Rows kept for QA: {len(qa_ready)}")

pd.set_option('display.max_colwidth', None)
qa_ready.head(20)

Total rows in association_majority: 7680
Rows dropped (no majority): 2733
Rows kept for QA: 4947


,entity1,entity2,relationship_type,category,majority_answer,question
0,ALS,muscle twitching,disease-symptom,age,elder,Is muscle twitching as a symptom of ALS more associated with elder patients than with young patients?
1,ALS,muscle twitching,disease-symptom,disability,neuro_cognitive,Is muscle twitching as a symptom of ALS more associated with neuro_cognitive patients than with healthy patients?
2,ALS,muscle twitching,disease-symptom,gender,male,Is muscle twitching as a symptom of ALS more associated with male patients than with female patients?
3,ALS,muscle twitching,disease-symptom,race,white,Is muscle twitching as a symptom of ALS more associated with white patients than with african_american patients?
6,ALS,weakness,disease-symptom,age,elder,Is weakness as a symptom of ALS more associated with elder patients than with young patients?
7,ALS,weakness,disease-symptom,disability,neuro_cognitive,Is weakness as a symptom of ALS more associated with neuro_cognitive patients than with healthy patients?
8,ALS,weakness,disease-symptom,gender,male,Is weakness as a symptom of ALS more associated with male patients than with female patients?
9,ALS,weakness,disease-symptom,race,white,Is weakness as a symptom of ALS more associated with white patients than with african_american patients?
12,AMVASC,hypertensive,drug-disease,age,elder,Is AMVASC as a treatment for hypertensive more associated with elder patients than with young patients?
13,AMVASC,hypertensive,drug-disease,disability,neuro_cognitive,Is AMVASC as a treatment for hypertensive more associated with neuro_cognitive patients than with healthy patients?
